In [95]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 10.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [175]:
import pandas as pd
import numpy as np

input_file = "/Users/mollyshallow/Desktop/Proxiome_plots/Proxiome_linescan_condensed2.xlsx"
output_file = "/Users/mollyshallow/Desktop/Proxiome_plots/Proxiome_linescan_condensed_centeraligned3.xlsx"

# Read all sheets
sheets = pd.read_excel(input_file, sheet_name=None, header=None)
aligned_sheets = {}

for sheet_name, df in sheets.items():
    print(f"\nProcessing sheet: {sheet_name}")
    df_out = df.copy(deep=True)

    # Drop the old label row (row 2, index 2)
    df_out = df_out.drop(index=2).reset_index(drop=True)

    # Row 2 (after dropping) now contains the actual distance column labels
    header_row = df_out.iloc[2].fillna("").astype(str).str.strip()
    print("Header row preview (after drop):", header_row.tolist()[:10])

    # Row 0 contains synapse IDs
    synapse_row = df_out.iloc[0].fillna("").astype(str).str.strip()

    # Identify distance columns using regex
    distance_cols = header_row[
        header_row.str.contains(r"distance", case=False, regex=True, na=False)
    ].index

    if len(distance_cols) == 0:
        print(f"No distance columns found in sheet: {sheet_name}, skipping.")
        continue

    print(f"Found distance columns: {list(distance_cols)}")

    # Loop over distance columns to center-align
    for dist_col in distance_cols:
        synapse_id = synapse_row[dist_col]

        # Columns belonging to this synapse
        synapse_cols = [c for c in df_out.columns if synapse_row[c] == synapse_id]

        # Original distances starting from row 3
        distances = pd.to_numeric(df_out.iloc[3:, dist_col], errors="coerce")

        if distances.notna().sum() == 0:
            continue

        # Compute median and subtract
        center = distances.median()
        df_out.iloc[3:, dist_col] = distances - center

        print(f"Synapse '{synapse_id}': median={center:.3f} centered.")

    aligned_sheets[sheet_name] = df_out

# Save results
if aligned_sheets:
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, df_out in aligned_sheets.items():
            df_out.to_excel(writer, sheet_name=sheet_name, index=False, header=False)
    print(f"\nCenter-aligned Excel file saved: {output_file}")
else:
    print("No sheets had distance columns. Nothing was saved.")

    


Processing sheet: ZO1 jam3 doub
Header row preview (after drop): ['Distance_(microns)', 'ZO1', 'Distance_(microns)', 'ZO1', 'Distance_(microns)', 'ZO1', 'Distance_(microns)', 'ZO1', 'Distance_(microns)', 'ZO1']
Found distance columns: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62]
Synapse '': median=1.677 centered.
Synapse '': median=1.479 centered.
Synapse '': median=1.696 centered.
Synapse '': median=1.461 centered.
Synapse '': median=1.659 centered.
Synapse '': median=1.623 centered.
Synapse '': median=1.714 centered.
Synapse '': median=1.948 centered.
Synapse '': median=1.641 centered.
Synapse '': median=1.623 centered.
Synapse '': median=1.479 centered.
Synapse '': median=1.804 centered.
Synapse '': median=1.569 centered.
Synapse '': median=1.659 centered.
Synapse '': median=1.533 centered.
Synapse '': median=1.479 centered.
Synapse '': median=1.587 centered.
Synapse '': median=1.605 centered.
Synapse ''

In [197]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =========================
# USER INPUT
# =========================
aligned_file = "/Users/mollyshallow/Desktop/Proxiome_plots/Proxiome_linescan_condensed_centeraligned3.xlsx"
output_folder = "/Users/mollyshallow/Desktop/Proxiome_plots"
SAVE_PLOTS = True  # Set to False to just show plots

CHANNEL_COLORS = {
    "Cx35": "tab:blue",
    "Cx34": "gold",
    "ZO1": "magenta"
}

# Create output folder if saving
if SAVE_PLOTS and not os.path.exists(output_folder):
    os.makedirs(output_folder)

# =========================
# LOAD SHEETS
# =========================
sheets = pd.read_excel(aligned_file, sheet_name=None, header=None)

# =========================
# HELPERS
# =========================
def mean_sem(aligned_list, grid):
    """Interpolate all columns onto a common grid and compute mean ± SEM."""
    interp_vals = []
    for df_ in aligned_list:
        valid = df_.dropna()
        interp = np.interp(
            grid,
            valid["aligned_distance"],
            valid["value"],
            left=np.nan,
            right=np.nan
        )
        interp_vals.append(interp)
    arr = np.vstack(interp_vals)
    mean = np.nanmean(arr, axis=0)
    sem = np.nanstd(arr, axis=0) / np.sqrt(np.sum(~np.isnan(arr), axis=0))
    return mean, sem

# =========================
# COLLECT DATA PER MARKER
# =========================
marker_plot_data = {marker: {} for marker in CHANNEL_COLORS}  # {marker: {sheet_name: (grid, mean, sem)}}

for sheet_name, df in sheets.items():
    header_row = (
        df.iloc[2]
        .astype(str)
        .str.replace("\xa0", "")
        .str.strip()
        .str.lower()
    )
    data = df.iloc[3:].copy()
    distance_cols_idx = header_row[header_row.str.contains("distance")].index.tolist()

    for marker, color in CHANNEL_COLORS.items():
        marker_lower = marker.lower()
        aligned_list = []

        # Find all marker columns
        marker_cols_idx = [i for i, h in enumerate(header_row) if marker_lower in h]
        for col_idx in marker_cols_idx:
            left_distance_idx = [i for i in distance_cols_idx if i < col_idx]
            if not left_distance_idx:
                continue
            distance_idx = max(left_distance_idx)

            distances = pd.to_numeric(data.iloc[:, distance_idx], errors="coerce")
            values    = pd.to_numeric(data.iloc[:, col_idx], errors="coerce")
            valid = values.notna()
            if valid.sum() == 0:
                continue

            aligned_list.append(pd.DataFrame({
                "aligned_distance": distances,
                "value": values
            }))

        if not aligned_list:
            continue

        # Compute grid, mean, sem
        all_dists = pd.concat(df_["aligned_distance"] for df_ in aligned_list)
        distance_grid = np.linspace(all_dists.min(), all_dists.max(), 500)
        mean, sem = mean_sem(aligned_list, distance_grid)

        marker_plot_data[marker][sheet_name] = (distance_grid, mean, sem)

# =========================
# PLOT PER MARKER
# =========================
for marker, sheet_data in marker_plot_data.items():
    n_sheets = len(sheet_data)
    fig, axes = plt.subplots(n_sheets, 1, figsize=(10, 4*n_sheets),
                             sharex=True, sharey=True)
    if n_sheets == 1:
        axes = [axes]

    for ax, (sheet_name, (grid, mean, sem)) in zip(axes, sheet_data.items()):
        ax.plot(grid, mean, label=marker, color=CHANNEL_COLORS[marker], linewidth=2)
        ax.fill_between(grid, mean - sem, mean + sem, color=CHANNEL_COLORS[marker], alpha=0.3)
        ax.set_title(sheet_name)
        ax.grid(False)
        # ax.axvline(x=0, color='r', linestyle='--')
        sns.despine(ax=ax)
        sns.set_style('ticks')
        ax.legend()

    # Shared axis labels
    axes[-1].set_xlabel("Distance from synapse center (µm)")
    fig.text(0.04, 0.5, "Background-subtracted gray value", va='center', rotation='vertical')

    # Global limits
    plt.xlim(-1.5, 1.5)
    plt.ylim(-0.2, 1.5)
    
    plt.tight_layout(rect=[0.05,0,1,1])
    # plt.show()

    # -------------------
    # SHOW OR SAVE
    # -------------------
    if SAVE_PLOTS:
        save_path = os.path.join(output_folder, f"{marker}_plots_v2.pdf")
        plt.savefig(save_path, dpi=300)
        plt.close(fig)
        print(f"Saved {marker} plot to: {save_path}")
    else:
        plt.show()


Saved Cx35 plot to: /Users/mollyshallow/Desktop/Proxiome_plots/Cx35_plots_v2.pdf
Saved Cx34 plot to: /Users/mollyshallow/Desktop/Proxiome_plots/Cx34_plots_v2.pdf
Saved ZO1 plot to: /Users/mollyshallow/Desktop/Proxiome_plots/ZO1_plots_v2.pdf


In [196]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.backends.backend_pdf import PdfPages
# =========================
# USER INPUT
# =========================
aligned_file = "/Users/mollyshallow/Desktop/Proxiome_plots/Proxiome_linescan_condensed_centeraligned3.xlsx"
output_folder = "/Users/mollyshallow/Desktop/Proxiome_plots"
SAVE_PLOTS = True  # Set to False to just show plots

CHANNEL_COLORS = {
    "Cx35": "tab:blue",
    "Cx34": "gold",
    "ZO1": "magenta"
}

# Create output folder if saving
if SAVE_PLOTS and not os.path.exists(output_folder):
    os.makedirs(output_folder)

# =========================
# LOAD SHEETS
# =========================
sheets = pd.read_excel(aligned_file, sheet_name=None, header=None)

# =========================
# HELPERS
# =========================
def mean_sem(aligned_list, grid):
    """Interpolate all columns onto a common grid and compute mean ± SEM."""
    interp_vals = []
    for df_ in aligned_list:
        valid = df_.dropna()
        interp = np.interp(
            grid,
            valid["aligned_distance"],
            valid["value"],
            left=np.nan,
            right=np.nan
        )
        interp_vals.append(interp)
    arr = np.vstack(interp_vals)
    mean = np.nanmean(arr, axis=0)
    sem = np.nanstd(arr, axis=0) / np.sqrt(np.sum(~np.isnan(arr), axis=0))
    return mean, sem

# =========================
# PLOT INDIVIDUAL SYNAPSES
# =========================

# =========================
# ONE MULTI-PAGE PDF PER SHEET
# =========================

for sheet_name, df in sheets.items():

    # ---------- detect header row ----------
    header_row_idx = None
    for i in range(5):
        row = df.iloc[i].astype(str).str.lower()
        if row.str.contains("distance").any():
            header_row_idx = i
            break
    if header_row_idx is None:
        raise ValueError(f"Could not find header row in sheet '{sheet_name}'")

    synapse_names = df.iloc[header_row_idx - 1].astype(str).fillna("").str.strip()

    header_row = (
        df.iloc[header_row_idx]
        .astype(str)
        .str.replace("\xa0", "")
        .str.strip()
        .str.lower()
    )

    data = df.iloc[header_row_idx + 1:].copy()

    distance_cols_idx = header_row[header_row.str.contains("distance")].index.tolist()

    # ---------- de-duplicate synapse names ----------
    # counts = {}
    # for i, name in synapse_names.items():
    #     if not name or name.lower() == "nan":
    #         continue
    #     if name in counts:
    #         counts[name] += 1
    #         synapse_names[i] = f"{name}_{counts[name]}"
    #     else:
    #         counts[name] = 1

    # ---------- open PDF ----------
    if SAVE_PLOTS:
        pdf_path = os.path.join(output_folder, f"{sheet_name}_individual_synapses.pdf")
        pdf = PdfPages(pdf_path)

    # =========================
    # PLOT EACH SYNAPSE
    # =========================
    for col_idx, header in enumerate(header_row):

        for marker, color in CHANNEL_COLORS.items():
            if marker.lower() not in header:
                continue

            left_distance_idx = [i for i in distance_cols_idx if i < col_idx]
            if not left_distance_idx:
                continue
            distance_idx = max(left_distance_idx)

            distances = pd.to_numeric(data.iloc[:, distance_idx], errors="coerce")
            values    = pd.to_numeric(data.iloc[:, col_idx], errors="coerce")

            valid = distances.notna() & values.notna()
            if valid.sum() == 0:
                continue

            synapse_name = synapse_names[col_idx]

            # ---------- plot ----------
            fig, ax = plt.subplots(figsize=(6, 4))

            ax.plot(
                distances[valid],
                values[valid],
                color=color,
                linewidth=2,
                label=marker
            )

            ax.set_title(f"{sheet_name}\n{synapse_name} – {marker}")
            ax.set_xlabel("Distance from synapse center (µm)")
            ax.set_ylabel("Background-subtracted gray value")

            ax.set_xlim(-1.5, 1.5)
            ax.set_ylim(-0.2, 1.5)

            sns.despine(ax=ax)
            ax.legend()

            fig.tight_layout()

            if SAVE_PLOTS:
                pdf.savefig(fig)
                plt.close(fig)
            else:
                plt.show()

    # ---------- close PDF ----------
    if SAVE_PLOTS:
        pdf.close()
        print(f"Saved multi-page PDF: {pdf_path}")


Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/ZO1 jam3 doub_individual_synapses.pdf
Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/ZO1 clmp doubs_individual_synapses.pdf
Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/ZO1 uninj_individual_synapses.pdf
Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/Cxs jam3 doub_individual_synapses.pdf
Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/Cxs clmp doub_individual_synapses.pdf
Saved multi-page PDF: /Users/mollyshallow/Desktop/Proxiome_plots/Cxs uninj_individual_synapses.pdf
